# 🛠️ Chapter 04-02b: Hands-On — Store & Search 1000 Documents

---

## From Arrays to Real Vector Databases! 🗄️

In this notebook, you'll use ChromaDB and FAISS to build scalable vector stores.

By the end:
- ✅ Built a ChromaDB collection with metadata
- ✅ Performed similarity search with filters
- ✅ Used FAISS for high-performance search
- ✅ Compared ChromaDB vs FAISS
- ✅ Stored and searched 1000 documents

Let's build! 🚀

---

## 📦 Setup

In [ ]:
# !pip install chromadb faiss-cpu openai python-dotenv langchain-openai langchain-community

In [ ]:
import os
import time
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

def get_embeddings(texts, model="text-embedding-3-small"):
    response = client.embeddings.create(model=model, input=texts)
    return [item.embedding for item in response.data]

print("✅ Ready!" if os.getenv("OPENAI_API_KEY") else "❌ Set OPENAI_API_KEY in .env")

---

## 🧪 Part 1: ChromaDB

In [ ]:
import chromadb

# Create a client (in-memory for this demo)
chroma_client = chromadb.Client()

# Create a collection
collection = chroma_client.create_collection(
    name="tech_docs",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity
)

print(f"✅ Collection created: {collection.name}")
print(f"   Count: {collection.count()}")

In [ ]:
# Add documents with metadata
documents = [
    "Python is a versatile programming language used in AI and web development.",
    "React is a JavaScript library for building interactive user interfaces.",
    "TensorFlow is an open-source framework for machine learning.",
    "Docker containers package applications with all their dependencies.",
    "PostgreSQL is a powerful open-source relational database.",
    "Kubernetes orchestrates containerized applications at scale.",
    "LangChain simplifies building applications with large language models.",
    "FastAPI is a modern Python web framework for building APIs.",
    "Redis is an in-memory data store used for caching.",
    "PyTorch is Facebook's deep learning framework.",
]

categories = ["language", "frontend", "ml", "devops", "database",
              "devops", "ai", "backend", "database", "ml"]

# ChromaDB can auto-embed or accept pre-computed embeddings
# Let's use OpenAI embeddings for quality
embeddings = get_embeddings(documents)

collection.add(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    embeddings=embeddings,
    metadatas=[{"category": cat, "index": i} for i, cat in enumerate(categories)]
)

print(f"✅ Added {collection.count()} documents!")

In [ ]:
# Similarity search
query = "How do I build AI applications?"
query_embedding = get_embeddings([query])

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print(f"🔍 Query: \"{query}\"\n")
for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    sim = 1 - dist  # ChromaDB returns distance, not similarity
    print(f"  [{sim:.4f}] ({meta['category']}) {doc}")

In [ ]:
# Filtered search — combine similarity + metadata
query = "What tools are available?"
query_embedding = get_embeddings([query])

# Only search in "devops" category
filtered_results = collection.query(
    query_embeddings=query_embedding,
    n_results=3,
    where={"category": "devops"}
)

print(f"🔍 Query: \"{query}\" (filtered: devops only)\n")
for doc, meta in zip(filtered_results['documents'][0], filtered_results['metadatas'][0]):
    print(f"  ({meta['category']}) {doc}")

print("\n💡 Only DevOps docs returned, even though other docs might be more similar overall!")

---

## 🧪 Part 2: FAISS

In [ ]:
import faiss

# Convert embeddings to numpy array
embedding_matrix = np.array(embeddings, dtype='float32')
dimension = embedding_matrix.shape[1]

# Create FAISS index (exact search)
index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print(f"✅ FAISS index created")
print(f"   Dimension: {dimension}")
print(f"   Vectors: {index.ntotal}")

In [ ]:
# Search with FAISS
query = "machine learning frameworks"
query_vec = np.array(get_embeddings([query]), dtype='float32')

# Search for top 3
distances, indices = index.search(query_vec, k=3)

print(f"🔍 Query: \"{query}\"\n")
for dist, idx in zip(distances[0], indices[0]):
    print(f"  [L2 dist: {dist:.4f}] {documents[idx]}")

print("\n💡 Lower L2 distance = more similar (opposite of cosine!)")

---

## 🚀 Part 3: Scale to 1000 Documents

In [ ]:
# Generate 1000 synthetic documents
topics = [
    "Python programming", "JavaScript development", "Machine learning",
    "Web development", "Database management", "Cloud computing",
    "DevOps practices", "Data science", "Cybersecurity", "Mobile development"
]

subtopics = [
    "best practices", "getting started", "advanced techniques",
    "common mistakes", "performance optimization", "real-world applications",
    "interview questions", "tools and frameworks", "design patterns", "debugging tips"
]

import random
random.seed(42)

large_docs = []
large_metas = []
for i in range(1000):
    topic = random.choice(topics)
    subtopic = random.choice(subtopics)
    doc = f"Guide to {subtopic} in {topic}. This covers essential concepts and practical examples for {topic.lower()} with focus on {subtopic}."
    large_docs.append(doc)
    large_metas.append({"topic": topic, "subtopic": subtopic, "index": i})

print(f"📝 Generated {len(large_docs)} documents")
print(f"   Sample: {large_docs[0]}")

In [ ]:
# Embed in batches (API limit: ~2048 texts per call)
batch_size = 100
all_embeddings = []

print("Embedding 1000 documents...")
start = time.time()

for i in range(0, len(large_docs), batch_size):
    batch = large_docs[i:i + batch_size]
    batch_embs = get_embeddings(batch)
    all_embeddings.extend(batch_embs)
    print(f"  Batch {i//batch_size + 1}/10 done")

elapsed = time.time() - start
print(f"\n✅ Embedded {len(all_embeddings)} docs in {elapsed:.1f}s")

In [ ]:
# Load into ChromaDB
large_collection = chroma_client.create_collection(name="large_docs", metadata={"hnsw:space": "cosine"})

# Add in batches
for i in range(0, len(large_docs), batch_size):
    end = min(i + batch_size, len(large_docs))
    large_collection.add(
        ids=[f"doc_{j}" for j in range(i, end)],
        documents=large_docs[i:end],
        embeddings=all_embeddings[i:end],
        metadatas=large_metas[i:end]
    )

print(f"✅ ChromaDB: {large_collection.count()} documents loaded")

# Also load into FAISS
large_matrix = np.array(all_embeddings, dtype='float32')
large_index = faiss.IndexFlatL2(large_matrix.shape[1])
large_index.add(large_matrix)

print(f"✅ FAISS: {large_index.ntotal} vectors indexed")

In [ ]:
# Speed comparison: ChromaDB vs FAISS on 1000 docs

query = "How to optimize machine learning model performance?"
query_emb = get_embeddings([query])

# ChromaDB search
start = time.time()
chroma_results = large_collection.query(query_embeddings=query_emb, n_results=5)
chroma_time = time.time() - start

# FAISS search
query_vec = np.array(query_emb, dtype='float32')
start = time.time()
distances, indices = large_index.search(query_vec, k=5)
faiss_time = time.time() - start

print(f"⏱️  ChromaDB: {chroma_time*1000:.2f}ms")
print(f"⏱️  FAISS:    {faiss_time*1000:.2f}ms")

print(f"\n🔍 ChromaDB Top 3:")
for doc, meta in zip(chroma_results['documents'][0][:3], chroma_results['metadatas'][0][:3]):
    print(f"  [{meta['topic']}] {doc[:60]}...")

print(f"\n🔍 FAISS Top 3:")
for idx in indices[0][:3]:
    print(f"  [{large_metas[idx]['topic']}] {large_docs[idx][:60]}...")

In [ ]:
# ChromaDB advantage: metadata filtering

query = "How to get started?"
query_emb = get_embeddings([query])

# Search only in "Machine learning" topic
filtered = large_collection.query(
    query_embeddings=query_emb,
    n_results=5,
    where={"topic": "Machine learning"}
)

print(f"🔍 '{query}' — filtered to Machine Learning only:\n")
for doc, meta in zip(filtered['documents'][0], filtered['metadatas'][0]):
    print(f"  [{meta['subtopic']}] {doc[:70]}...")

print("\n💡 FAISS can't do this without a separate metadata store!")

---

## ✍️ Practice: LangChain Vector Store Integration

**Challenge:** Use LangChain's ChromaDB wrapper to build a vector store.

In [ ]:
# YOUR CODE HERE

# from langchain_openai import OpenAIEmbeddings
# from langchain_community.vectorstores import Chroma

# TODO: Create an OpenAIEmbeddings instance
# TODO: Create a Chroma vector store from documents using Chroma.from_texts()
# TODO: Use similarity_search() to find relevant docs
# TODO: Use similarity_search_with_score() to see distances

<details>
<summary>💡 Click to see solution</summary>

```python
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_texts(
    texts=documents,
    embedding=embeddings_model,
    metadatas=[{"category": c} for c in categories]
)

# Simple search
results = vectorstore.similarity_search("AI frameworks", k=3)
for r in results:
    print(f"  {r.page_content}")

# With scores
results_with_scores = vectorstore.similarity_search_with_score("AI frameworks", k=3)
for doc, score in results_with_scores:
    print(f"  [{score:.4f}] {doc.page_content}")
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 ChromaDB: add(), query(), with metadata filtering. Easy!       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 FAISS: raw speed, no frills. Great for benchmarking.           │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Batch your embeddings — one API call per batch of 100.         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Metadata filters are critical for production RAG.              │
├─────────────────────────────────────────────────────────────────────┤
│  📌 LangChain wraps all vector DBs with a unified interface.       │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: RAG — Retrieval-Augmented Generation!

We'll combine everything: embeddings + vector stores + LLM = a chatbot that knows YOUR data.

---

### 🎉 You Built Real Vector Stores!
ChromaDB, FAISS, 1000 documents, metadata filtering — production patterns! 🚀